
# Buy Fee Model V14 — Range-Based Pricing Decision Tool

### Goal
Create a practical **directional decision-support tool** for Pricing using the evidence already developed in V8/V10/V11/V13.

For every observed **Location × Lane × Price Tier** and each scenario **+$10 / +$15 / +$20**, V14 reports:

- RPU direction
- empirical RPU range
- Conversion downside range
- Volume downside range
- downside-risk flags
- evidence strength
- number of observations / independent change dates
- the **evidence hierarchy level actually used**

### Evidence hierarchy
V14 does not require repeated observations at the exact segment. It backs off conservatively:

1. **Exact Segment:** Location × Lane × Price Tier
2. **Location × Price Tier**
3. **Lane × Price Tier**
4. **Price Tier**
5. **Location**
6. **Portfolio Scenario**

A level is used only when it has enough historical support. Otherwise V14 moves to the next broader level.

### Interpretation
This is not an automated pricing engine and does not claim exact causal forecasts. It answers:

> Based on the closest historical evidence available, what direction and range of outcomes should Pricing consider if this fee scenario is applied?


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"
V10_DIR = DATA_FOLDER / "buy_fee_model_v10_1_outputs"
V11_DIR = DATA_FOLDER / "buy_fee_model_v11_outputs"
V13_DIR = DATA_FOLDER / "buy_fee_model_v13_outputs"

V8_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"
V11_SEGMENT_FILE = V11_DIR / "07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv"
V11_VALIDATION_FILE = V11_DIR / "04_SCENARIO_MODEL_VALIDATION.csv"
V13_DECISION_FILE = V13_DIR / "02_V13_SEGMENT_DECISION_TABLE.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v14_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20]
TOLERANCE = 2.5

# Evidence minimums.
# Broader levels require more support because they are less segment-specific.
MIN_SUPPORT = {
    "Exact Segment": {"events": 3, "dates": 2},
    "Location x Price Tier": {"events": 5, "dates": 2},
    "Lane x Price Tier": {"events": 8, "dates": 3},
    "Price Tier": {"events": 12, "dates": 4},
    "Location": {"events": 12, "dates": 4},
    "Portfolio Scenario": {"events": 25, "dates": 5},
}

# Direction thresholds: intentionally modest because this is directional.
RPU_POSITIVE_THRESHOLD = 2.0
RPU_NEGATIVE_THRESHOLD = -2.0

print("="*80)
print("V14 INPUT CHECK")
print("="*80)
for p in [V8_FILE, V11_SEGMENT_FILE, V11_VALIDATION_FILE, V13_DECISION_FILE]:
    print(f"{p.name:<48} {p.exists()}")

if not V8_FILE.exists():
    raise FileNotFoundError(f"Required V8 file missing: {V8_FILE}")

print("\nOutput:", OUTPUT_DIR)


V14 INPUT CHECK
04_DID_EVENT_LEVEL_RESULTS.csv                   True
07_SEGMENT_RESPONSE_WITH_CONFIDENCE.csv          True
04_SCENARIO_MODEL_VALIDATION.csv                 True
02_V13_SEGMENT_DECISION_TABLE.csv                True

Output: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v14_outputs


## 1. Load and normalize V8 empirical event evidence

In [2]:

v8 = pd.read_csv(V8_FILE, low_memory=True)

def first_existing(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            "Could not find any of these columns:\n" +
            "\n".join(candidates)
        )
    return None

COL = {
    "event": first_existing(v8, ["Segment_Change_ID"]),
    "date": first_existing(v8, ["Corporate_Change_ID"]),
    "fee": first_existing(v8, ["BUYFEE_CHANGE_AMOUNT"]),
    "location": first_existing(v8, ["Treated_Location", "Location"]),
    "lane": first_existing(v8, ["Treated_Lane", "Lane"]),
    "tier": first_existing(v8, ["Treated_Price_Tier", "Price_Tier", "Price Tier"]),
    "rpu": first_existing(v8, ["DiD_RPU_Change_Dollar"]),
    "conv": first_existing(v8, ["DiD_Conversion_Change_pp"]),
    "vol": first_existing(v8, ["DiD_Volume_Change_Pct"]),
}

events = pd.DataFrame({
    "Event_ID": v8[COL["event"]],
    "Corporate_Change_ID": v8[COL["date"]],
    "Fee_Change": pd.to_numeric(v8[COL["fee"]], errors="coerce"),
    "Location": v8[COL["location"]].astype("string").fillna("").str.strip().str.upper(),
    "Lane": v8[COL["lane"]].astype("string").fillna("").str.strip().str.upper(),
    "Price_Tier": v8[COL["tier"]].astype("string").fillna("").str.strip(),
    "RPU_DiD": pd.to_numeric(v8[COL["rpu"]], errors="coerce"),
    "Conversion_DiD_pp": pd.to_numeric(v8[COL["conv"]], errors="coerce"),
    "Volume_DiD_Pct": pd.to_numeric(v8[COL["vol"]], errors="coerce"),
})

events = events[
    events["Location"].ne("") &
    events["Lane"].ne("") &
    events["Price_Tier"].ne("")
].copy()

def assign_scenario(x):
    if pd.isna(x):
        return np.nan
    candidates = [s for s in SCENARIOS if abs(x-s) <= TOLERANCE]
    return min(candidates, key=lambda s: abs(x-s)) if candidates else np.nan

events["Scenario"] = events["Fee_Change"].apply(assign_scenario)
events = events[events["Scenario"].notna()].copy()
events["Scenario"] = events["Scenario"].astype(int)

print("Scenario events:", f"{len(events):,}")
print("Locations:", events["Location"].nunique())
print("Exact segments:", events[["Location","Lane","Price_Tier"]].drop_duplicates().shape[0])
print("Independent change dates:", events["Corporate_Change_ID"].nunique())
display(events.head())


Scenario events: 810
Locations: 24
Exact segments: 509
Independent change dates: 29


,Event_ID,Corporate_Change_ID,Fee_Change,Location,Lane,Price_Tier,RPU_DiD,Conversion_DiD_pp,Volume_DiD_Pct,Scenario
20,ATLANTA_C_0-1000_2024-12-01,2024-12-01,15.0,ATLANTA,C,0-1000,0.415501,-3.802852,0.183838,15
41,ATLANTA_E_0-1000_2024-12-01,2024-12-01,20.0,ATLANTA,E,0-1000,8.344600,-30.127789,-0.392741,20
56,ATLANTA_IL_0-1000_2024-12-01,2024-12-01,10.0,ATLANTA,IL,0-1000,-40.020588,-1.363501,0.129292,10
58,ATLANTA_IO_0-1000_2024-12-01,2024-12-01,10.0,ATLANTA,IO,0-1000,32.550977,6.511097,-0.023973,10
131,BATONROUGE_1_1000-2000_2024-12-01,2024-12-01,10.0,BATONROUGE,1,1000-2000,-61.805607,-3.886555,0.307822,10


## 2. Create the complete Location × Lane × Tier × scenario decision grid

In [3]:

segment_universe = (
    events[["Location","Lane","Price_Tier"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

scenario_df = pd.DataFrame({"Scenario": SCENARIOS})

segment_universe["_key"] = 1
scenario_df["_key"] = 1

grid = (
    segment_universe
    .merge(scenario_df, on="_key")
    .drop(columns="_key")
)

print("Segments:", len(segment_universe))
print("Decision rows:", len(grid))
display(grid.head(10))


Segments: 509
Decision rows: 1527


,Location,Lane,Price_Tier,Scenario
0,ATLANTA,C,0-1000,10
1,ATLANTA,C,0-1000,15
2,ATLANTA,C,0-1000,20
3,ATLANTA,E,0-1000,10
4,ATLANTA,E,0-1000,15
5,ATLANTA,E,0-1000,20
6,ATLANTA,IL,0-1000,10
7,ATLANTA,IL,0-1000,15
8,ATLANTA,IL,0-1000,20
9,ATLANTA,IO,0-1000,10


## 3. Empirical evidence summary function

In [4]:

def evidence_stats(d):
    if d.empty:
        return None

    rpu = d["RPU_DiD"].dropna()
    conv = d["Conversion_DiD_pp"].dropna()
    vol = d["Volume_DiD_Pct"].dropna()

    return {
        "Evidence_Events": int(len(d)),
        "Evidence_Change_Dates": int(d["Corporate_Change_ID"].nunique()),
        "Evidence_Locations": int(d["Location"].nunique()),

        # RPU central tendency and empirical middle 80% range
        "RPU_Median": rpu.median() if len(rpu) else np.nan,
        "RPU_P10": rpu.quantile(.10) if len(rpu) else np.nan,
        "RPU_P25": rpu.quantile(.25) if len(rpu) else np.nan,
        "RPU_P75": rpu.quantile(.75) if len(rpu) else np.nan,
        "RPU_P90": rpu.quantile(.90) if len(rpu) else np.nan,
        "RPU_Positive_Rate": (rpu > 0).mean() if len(rpu) else np.nan,

        # Conversion empirical range / downside frequencies
        "Conversion_Median_pp": conv.median() if len(conv) else np.nan,
        "Conversion_P10_pp": conv.quantile(.10) if len(conv) else np.nan,
        "Conversion_P90_pp": conv.quantile(.90) if len(conv) else np.nan,
        "Conversion_Down_GT_1pp_Rate": (conv < -1).mean() if len(conv) else np.nan,
        "Conversion_Down_GT_2pp_Rate": (conv < -2).mean() if len(conv) else np.nan,
        "Conversion_Down_GT_5pp_Rate": (conv < -5).mean() if len(conv) else np.nan,

        # Volume empirical range / downside frequencies
        "Volume_Median_Pct": vol.median() if len(vol) else np.nan,
        "Volume_P10_Pct": vol.quantile(.10) if len(vol) else np.nan,
        "Volume_P90_Pct": vol.quantile(.90) if len(vol) else np.nan,
        "Volume_Down_GT_3pct_Rate": (vol < -.03).mean() if len(vol) else np.nan,
        "Volume_Down_GT_10pct_Rate": (vol < -.10).mean() if len(vol) else np.nan,
        "Volume_Down_GT_20pct_Rate": (vol < -.20).mean() if len(vol) else np.nan,
    }

def enough_support(stats, level):
    if stats is None:
        return False
    req = MIN_SUPPORT[level]
    return (
        stats["Evidence_Events"] >= req["events"]
        and stats["Evidence_Change_Dates"] >= req["dates"]
    )


## 4. Hierarchical fallback engine

In [5]:

def subset_for_level(row, level):
    d = events[events["Scenario"].eq(row["Scenario"])]

    if level == "Exact Segment":
        return d[
            d["Location"].eq(row["Location"]) &
            d["Lane"].eq(row["Lane"]) &
            d["Price_Tier"].eq(row["Price_Tier"])
        ]

    if level == "Location x Price Tier":
        return d[
            d["Location"].eq(row["Location"]) &
            d["Price_Tier"].eq(row["Price_Tier"])
        ]

    if level == "Lane x Price Tier":
        return d[
            d["Lane"].eq(row["Lane"]) &
            d["Price_Tier"].eq(row["Price_Tier"])
        ]

    if level == "Price Tier":
        return d[
            d["Price_Tier"].eq(row["Price_Tier"])
        ]

    if level == "Location":
        return d[
            d["Location"].eq(row["Location"])
        ]

    return d

HIERARCHY = [
    "Exact Segment",
    "Location x Price Tier",
    "Lane x Price Tier",
    "Price Tier",
    "Location",
    "Portfolio Scenario",
]

def choose_evidence(row):
    fallback_records = []

    for level in HIERARCHY:
        d = subset_for_level(row, level)
        stats = evidence_stats(d)

        if stats is not None:
            fallback_records.append(
                f"{level}: {stats['Evidence_Events']} events / "
                f"{stats['Evidence_Change_Dates']} dates"
            )

        if enough_support(stats, level):
            return level, stats, " -> ".join(fallback_records)

    # Safety: if even portfolio is thin, still return portfolio evidence
    d = subset_for_level(row, "Portfolio Scenario")
    stats = evidence_stats(d)
    return "Portfolio Scenario - Thin", stats, " -> ".join(fallback_records)

records = []

for i, row in grid.iterrows():
    level, stats, trail = choose_evidence(row)

    rec = row.to_dict()
    rec["Evidence_Level_Used"] = level
    rec["Evidence_Search_Trail"] = trail

    if stats:
        rec.update(stats)

    records.append(rec)

    if (i+1) % 500 == 0:
        print(f"Processed {i+1:,} / {len(grid):,}")

decision = pd.DataFrame(records)

print("Decision rows:", len(decision))
print("\nEvidence level usage:")
display(
    decision["Evidence_Level_Used"]
    .value_counts(dropna=False)
    .rename_axis("Evidence_Level")
    .reset_index(name="Rows")
)


Processed 500 / 1,527
Processed 1,000 / 1,527
Processed 1,500 / 1,527
Decision rows: 1527

Evidence level usage:


,Evidence_Level,Rows
0,Portfolio Scenario,830
1,Price Tier,436
2,Location x Price Tier,188
3,Exact Segment,37
4,Location,27
5,Lane x Price Tier,9


## 5. Direction, downside risk, and evidence strength

In [6]:

def rpu_direction(r):
    med = r["RPU_Median"]
    pos = r["RPU_Positive_Rate"]

    if pd.isna(med) or pd.isna(pos):
        return "Insufficient Evidence"

    if med >= RPU_POSITIVE_THRESHOLD and pos >= .60:
        return "Likely Increase"

    if med <= RPU_NEGATIVE_THRESHOLD and pos <= .40:
        return "Likely Decrease"

    if med > 0 and pos >= .50:
        return "Leans Increase"

    if med < 0 and pos < .50:
        return "Leans Decrease"

    return "Uncertain"


def conversion_risk(r):
    p2 = r["Conversion_Down_GT_2pp_Rate"]
    p5 = r["Conversion_Down_GT_5pp_Rate"]

    if pd.isna(p2):
        return "Unknown"

    if p2 <= .25 and (pd.isna(p5) or p5 <= .15):
        return "Low"

    if p2 <= .45:
        return "Moderate"

    return "High"


def volume_risk(r):
    p10 = r["Volume_Down_GT_10pct_Rate"]
    p20 = r["Volume_Down_GT_20pct_Rate"]

    if pd.isna(p10):
        return "Unknown"

    if p10 <= .25 and (pd.isna(p20) or p20 <= .15):
        return "Low"

    if p10 <= .45:
        return "Moderate"

    return "High"


def evidence_strength(r):
    n = r["Evidence_Events"]
    dates = r["Evidence_Change_Dates"]
    level = str(r["Evidence_Level_Used"])

    if pd.isna(n) or pd.isna(dates):
        return "Insufficient"

    # Exact / near-exact evidence gets credit for specificity.
    if level == "Exact Segment" and n >= 8 and dates >= 3:
        return "Strong"

    if level == "Location x Price Tier" and n >= 10 and dates >= 3:
        return "Strong"

    # Broader evidence can still be strong when replicated.
    if n >= 25 and dates >= 8:
        return "Strong"

    if n >= 10 and dates >= 3:
        return "Moderate"

    if n >= 3 and dates >= 2:
        return "Limited"

    return "Insufficient"


decision["RPU_Direction"] = decision.apply(rpu_direction, axis=1)
decision["Conversion_Risk"] = decision.apply(conversion_risk, axis=1)
decision["Volume_Risk"] = decision.apply(volume_risk, axis=1)
decision["Evidence_Strength"] = decision.apply(evidence_strength, axis=1)

decision["RPU_Range_Crosses_Zero"] = (
    (decision["RPU_P10"] <= 0) &
    (decision["RPU_P90"] >= 0)
)

decision["Downside_Flag"] = np.select(
    [
        (decision["Conversion_Risk"] == "High") |
        (decision["Volume_Risk"] == "High"),

        (decision["Conversion_Risk"] == "Moderate") |
        (decision["Volume_Risk"] == "Moderate"),
    ],
    [
        "High Downside Risk",
        "Monitor Downside"
    ],
    default="Lower Observed Downside"
)


## 6. Add V11/V13 context when available

In [7]:

# These enrich the output but never determine the empirical ranges.

if V11_VALIDATION_FILE.exists():
    val = pd.read_csv(V11_VALIDATION_FILE, low_memory=True)

    if "Fee_Increase" in val.columns and "Target" in val.columns:
        val["Fee_Increase"] = pd.to_numeric(val["Fee_Increase"], errors="coerce")

        rpu_val = val[
            val["Fee_Increase"].isin(SCENARIOS) &
            val["Target"].astype(str).str.upper().eq("RPU")
        ].copy()

        keep = [
            c for c in [
                "Fee_Increase",
                "Median_R2",
                "Median_MAE",
                "Median_Direction_Accuracy",
                "Folds"
            ]
            if c in rpu_val.columns
        ]

        if keep:
            rpu_val = rpu_val[keep].drop_duplicates("Fee_Increase")
            rpu_val = rpu_val.rename(columns={"Fee_Increase":"Scenario"})
            decision = decision.merge(rpu_val, on="Scenario", how="left")

if V13_DECISION_FILE.exists():
    v13 = pd.read_csv(V13_DECISION_FILE, low_memory=True)

    key_map = {
        "Fee_Increase":"Scenario",
        "Treated_Location":"Location",
        "Treated_Lane":"Lane",
        "Treated_Price_Tier":"Price_Tier",
    }

    if all(k in v13.columns for k in key_map):
        v13 = v13.rename(columns=key_map)

        optional = [
            c for c in [
                "Scenario","Location","Lane","Price_Tier",
                "Segment_Confidence",
                "Prediction_Quality_Flag"
            ]
            if c in v13.columns
        ]

        v13 = v13[optional].drop_duplicates(
            [c for c in ["Scenario","Location","Lane","Price_Tier"] if c in optional]
        )

        if len(optional) >= 4:
            decision = decision.merge(
                v13,
                on=["Scenario","Location","Lane","Price_Tier"],
                how="left"
            )

print("Optional context merged successfully.")


Optional context merged successfully.


## 7. Pricing-facing interpretation

In [8]:

def pricing_interpretation(r):
    direction = r["RPU_Direction"]
    crisk = r["Conversion_Risk"]
    vrisk = r["Volume_Risk"]
    strength = r["Evidence_Strength"]

    if strength == "Insufficient":
        return "Use only as broad directional context; historical support is thin."

    if direction in ["Likely Increase", "Leans Increase"]:
        if crisk == "High" or vrisk == "High":
            return "RPU evidence is favorable, but historical downside warrants close monitoring."
        if crisk == "Moderate" or vrisk == "Moderate":
            return "RPU evidence is favorable with meaningful downside uncertainty; suitable for controlled review."
        return "Historical evidence favors higher RPU with comparatively lower observed downside."

    if direction in ["Likely Decrease", "Leans Decrease"]:
        return "Historical evidence does not favor this scenario for RPU."

    return "Historical RPU direction is mixed; use as a sensitivity case rather than a primary signal."


decision["Pricing_Interpretation"] = decision.apply(
    pricing_interpretation,
    axis=1
)


## 8. Final V14 decision table

In [9]:

final_cols = [
    "Location","Lane","Price_Tier","Scenario",

    "RPU_Direction",
    "RPU_Median","RPU_P10","RPU_P90",
    "RPU_P25","RPU_P75","RPU_Positive_Rate",
    "RPU_Range_Crosses_Zero",

    "Conversion_Median_pp",
    "Conversion_P10_pp","Conversion_P90_pp",
    "Conversion_Down_GT_1pp_Rate",
    "Conversion_Down_GT_2pp_Rate",
    "Conversion_Down_GT_5pp_Rate",
    "Conversion_Risk",

    "Volume_Median_Pct",
    "Volume_P10_Pct","Volume_P90_Pct",
    "Volume_Down_GT_3pct_Rate",
    "Volume_Down_GT_10pct_Rate",
    "Volume_Down_GT_20pct_Rate",
    "Volume_Risk",

    "Downside_Flag",
    "Evidence_Strength",
    "Evidence_Level_Used",
    "Evidence_Events",
    "Evidence_Change_Dates",
    "Evidence_Locations",
    "Evidence_Search_Trail",

    "Pricing_Interpretation",

    "Median_R2","Median_MAE",
    "Median_Direction_Accuracy",
    "Folds",
    "Segment_Confidence",
    "Prediction_Quality_Flag",
]

final_cols = [c for c in final_cols if c in decision.columns]

final = decision[final_cols].copy()

final.to_csv(
    OUTPUT_DIR / "01_V14_PRICING_DECISION_TABLE.csv",
    index=False
)

print("Saved:", OUTPUT_DIR / "01_V14_PRICING_DECISION_TABLE.csv")
print("Rows:", len(final))
display(final.head(25))


Saved: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v14_outputs\01_V14_PRICING_DECISION_TABLE.csv
Rows: 1527


,Location,Lane,Price_Tier,Scenario,RPU_Direction,RPU_Median,RPU_P10,RPU_P90,RPU_P25,RPU_P75,...,Evidence_Change_Dates,Evidence_Locations,Evidence_Search_Trail,Pricing_Interpretation,Median_R2,Median_MAE,Median_Direction_Accuracy,Folds,Segment_Confidence,Prediction_Quality_Flag
0,ATLANTA,C,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-18.317166,8.716973,...,11,10,Location x Price Tier: 2 events / 1 dates -> L...,Historical evidence does not favor this scenar...,-0.411441,31.497519,0.642857,9,NaN,NaN
1,ATLANTA,C,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-11.746788,27.911824,...,10,14,Exact Segment: 1 events / 1 dates -> Location ...,RPU evidence is favorable with meaningful down...,-0.683987,34.239182,0.485714,1,Low,Weak
2,ATLANTA,C,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,1.208639,54.571618,...,10,12,Location x Price Tier: 1 events / 1 dates -> L...,RPU evidence is favorable with meaningful down...,-0.160923,20.850250,0.777941,2,NaN,NaN
3,ATLANTA,E,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-18.317166,8.716973,...,11,10,Location x Price Tier: 2 events / 1 dates -> P...,Historical evidence does not favor this scenar...,-0.411441,31.497519,0.642857,9,NaN,NaN
4,ATLANTA,E,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-11.746788,27.911824,...,10,14,Location x Price Tier: 1 events / 1 dates -> P...,RPU evidence is favorable with meaningful down...,-0.683987,34.239182,0.485714,1,NaN,NaN
5,ATLANTA,E,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,1.208639,54.571618,...,10,12,Exact Segment: 1 events / 1 dates -> Location ...,RPU evidence is favorable with meaningful down...,-0.160923,20.850250,0.777941,2,Low,Directional Only
6,ATLANTA,IL,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-18.317166,8.716973,...,11,10,Exact Segment: 1 events / 1 dates -> Location ...,Historical evidence does not favor this scenar...,-0.411441,31.497519,0.642857,9,Low,Directional Only
7,ATLANTA,IL,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-11.746788,27.911824,...,10,14,Location x Price Tier: 1 events / 1 dates -> P...,RPU evidence is favorable with meaningful down...,-0.683987,34.239182,0.485714,1,NaN,NaN
8,ATLANTA,IL,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,1.208639,54.571618,...,10,12,Location x Price Tier: 1 events / 1 dates -> P...,RPU evidence is favorable with meaningful down...,-0.160923,20.850250,0.777941,2,NaN,NaN
9,ATLANTA,IO,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-18.317166,8.716973,...,11,10,Exact Segment: 1 events / 1 dates -> Location ...,Historical evidence does not favor this scenar...,-0.411441,31.497519,0.642857,9,Low,Directional Only


## 9. Scenario summary

In [10]:

scenario_summary = (
    final.groupby("Scenario", as_index=False)
    .agg(
        Decision_Rows=("Location","size"),
        Locations=("Location","nunique"),
        Likely_or_Leans_RPU_Increase=(
            "RPU_Direction",
            lambda s: s.isin(["Likely Increase","Leans Increase"]).mean()
        ),
        Median_RPU=("RPU_Median","median"),
        Median_RPU_Low=("RPU_P10","median"),
        Median_RPU_High=("RPU_P90","median"),
        Low_Conversion_Risk_Share=(
            "Conversion_Risk",
            lambda s: (s=="Low").mean()
        ),
        High_Conversion_Risk_Share=(
            "Conversion_Risk",
            lambda s: (s=="High").mean()
        ),
        Low_Volume_Risk_Share=(
            "Volume_Risk",
            lambda s: (s=="Low").mean()
        ),
        High_Volume_Risk_Share=(
            "Volume_Risk",
            lambda s: (s=="High").mean()
        ),
        Strong_Evidence_Share=(
            "Evidence_Strength",
            lambda s: (s=="Strong").mean()
        ),
        Moderate_Evidence_Share=(
            "Evidence_Strength",
            lambda s: (s=="Moderate").mean()
        ),
        Limited_or_Insufficient_Share=(
            "Evidence_Strength",
            lambda s: s.isin(["Limited","Insufficient"]).mean()
        ),
    )
)

scenario_summary.to_csv(
    OUTPUT_DIR / "02_V14_SCENARIO_SUMMARY.csv",
    index=False
)

display(scenario_summary)


,Scenario,Decision_Rows,Locations,Likely_or_Leans_RPU_Increase,Median_RPU,Median_RPU_Low,Median_RPU_High,Low_Conversion_Risk_Share,High_Conversion_Risk_Share,Low_Volume_Risk_Share,High_Volume_Risk_Share,Strong_Evidence_Share,Moderate_Evidence_Share,Limited_or_Insufficient_Share
0,10,509,24,0.805501,9.141934,-31.285052,49.543294,0.074656,0.210216,0.039293,0.284872,0.624754,0.056974,0.318271
1,15,509,24,1.000000,11.170682,-42.530453,42.634438,0.000000,0.000000,0.000000,0.000000,0.866405,0.133595,0.000000
2,20,509,24,1.000000,24.466728,-28.308010,102.750672,0.000000,0.037328,0.017682,0.039293,0.726916,0.218075,0.055010


## 10. Evidence hierarchy usage

In [11]:

hierarchy_summary = (
    final.groupby(
        ["Scenario","Evidence_Level_Used","Evidence_Strength"],
        dropna=False
    )
    .size()
    .reset_index(name="Decision_Rows")
)

hierarchy_summary["Share_Within_Scenario"] = (
    hierarchy_summary["Decision_Rows"] /
    hierarchy_summary.groupby("Scenario")["Decision_Rows"].transform("sum")
)

hierarchy_summary.to_csv(
    OUTPUT_DIR / "03_V14_EVIDENCE_HIERARCHY_USAGE.csv",
    index=False
)

display(hierarchy_summary)


,Scenario,Evidence_Level_Used,Evidence_Strength,Decision_Rows,Share_Within_Scenario
0,10,Exact Segment,Limited,36,0.070727
1,10,Lane x Price Tier,Limited,9,0.017682
2,10,Location,Moderate,8,0.015717
3,10,Location x Price Tier,Limited,117,0.229862
4,10,Location x Price Tier,Strong,44,0.086444
5,10,Portfolio Scenario,Strong,19,0.037328
6,10,Price Tier,Moderate,21,0.041257
7,10,Price Tier,Strong,255,0.500982
8,15,Portfolio Scenario,Strong,441,0.866405
9,15,Price Tier,Moderate,68,0.133595


## 11. Pricing review table

In [12]:

# Pricing-facing subset.
# No automatic recommendation is made; this organizes the evidence for review.

pricing_review = final[
    [
        "Location","Lane","Price_Tier","Scenario",
        "RPU_Direction","RPU_Median","RPU_P10","RPU_P90",
        "Conversion_P10_pp","Conversion_P90_pp","Conversion_Risk",
        "Volume_P10_Pct","Volume_P90_Pct","Volume_Risk",
        "Downside_Flag",
        "Evidence_Strength","Evidence_Level_Used",
        "Evidence_Events","Evidence_Change_Dates",
        "Pricing_Interpretation"
    ]
].copy()

pricing_review.to_csv(
    OUTPUT_DIR / "04_V14_PRICING_REVIEW_TABLE.csv",
    index=False
)

display(pricing_review.head(30))


,Location,Lane,Price_Tier,Scenario,RPU_Direction,RPU_Median,RPU_P10,RPU_P90,Conversion_P10_pp,Conversion_P90_pp,Conversion_Risk,Volume_P10_Pct,Volume_P90_Pct,Volume_Risk,Downside_Flag,Evidence_Strength,Evidence_Level_Used,Evidence_Events,Evidence_Change_Dates,Pricing_Interpretation
0,ATLANTA,C,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-17.780424,22.018180,Moderate,-0.272407,0.665810,Moderate,Monitor Downside,Strong,Price Tier,51,11,Historical evidence does not favor this scenar...
1,ATLANTA,C,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-14.867319,19.312981,Moderate,-0.417839,0.512959,Moderate,Monitor Downside,Strong,Portfolio Scenario,83,10,RPU evidence is favorable with meaningful down...
2,ATLANTA,C,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,-15.285460,19.636977,Moderate,-0.438857,0.421232,Moderate,Monitor Downside,Strong,Portfolio Scenario,133,10,RPU evidence is favorable with meaningful down...
3,ATLANTA,E,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-17.780424,22.018180,Moderate,-0.272407,0.665810,Moderate,Monitor Downside,Strong,Price Tier,51,11,Historical evidence does not favor this scenar...
4,ATLANTA,E,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-14.867319,19.312981,Moderate,-0.417839,0.512959,Moderate,Monitor Downside,Strong,Portfolio Scenario,83,10,RPU evidence is favorable with meaningful down...
5,ATLANTA,E,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,-15.285460,19.636977,Moderate,-0.438857,0.421232,Moderate,Monitor Downside,Strong,Portfolio Scenario,133,10,RPU evidence is favorable with meaningful down...
6,ATLANTA,IL,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-17.780424,22.018180,Moderate,-0.272407,0.665810,Moderate,Monitor Downside,Strong,Price Tier,51,11,Historical evidence does not favor this scenar...
7,ATLANTA,IL,0-1000,15,Likely Increase,11.170682,-42.530453,42.634438,-14.867319,19.312981,Moderate,-0.417839,0.512959,Moderate,Monitor Downside,Strong,Portfolio Scenario,83,10,RPU evidence is favorable with meaningful down...
8,ATLANTA,IL,0-1000,20,Likely Increase,24.466728,-28.308010,102.750672,-15.285460,19.636977,Moderate,-0.438857,0.421232,Moderate,Monitor Downside,Strong,Portfolio Scenario,133,10,RPU evidence is favorable with meaningful down...
9,ATLANTA,IO,0-1000,10,Leans Decrease,-1.839170,-43.669269,26.313319,-17.780424,22.018180,Moderate,-0.272407,0.665810,Moderate,Monitor Downside,Strong,Price Tier,51,11,Historical evidence does not favor this scenar...


## 12. Data-quality / model limitations report

In [13]:

limitations = pd.DataFrame([
    {
        "Area":"Purpose",
        "V14_Treatment":"Directional decision support; not an exact fee elasticity forecast."
    },
    {
        "Area":"RPU",
        "V14_Treatment":"Uses empirical matched-control outcome distributions and reports ranges."
    },
    {
        "Area":"Conversion",
        "V14_Treatment":"Uses observed downside frequencies and empirical ranges as guardrails."
    },
    {
        "Area":"Volume",
        "V14_Treatment":"Uses observed downside frequencies and empirical ranges as guardrails."
    },
    {
        "Area":"Sparse exact segments",
        "V14_Treatment":"Automatically backs off through a transparent evidence hierarchy."
    },
    {
        "Area":"V11 predictive model",
        "V14_Treatment":"Validation is shown as context only; weak point-prediction accuracy does not drive empirical ranges."
    },
    {
        "Area":"Buyer data",
        "V14_Treatment":"Not used as event-time evidence because available buyer history ends before 2023 fee events."
    },
    {
        "Area":"Future updates",
        "V14_Treatment":"New cleaner events can be appended to V8-style event evidence; ranges and evidence strength will update."
    },
])

limitations.to_csv(
    OUTPUT_DIR / "05_V14_MODEL_LIMITATIONS.csv",
    index=False
)

display(limitations)


,Area,V14_Treatment
0,Purpose,Directional decision support; not an exact fee...
1,RPU,Uses empirical matched-control outcome distrib...
2,Conversion,Uses observed downside frequencies and empiric...
3,Volume,Uses observed downside frequencies and empiric...
4,Sparse exact segments,Automatically backs off through a transparent ...
5,V11 predictive model,Validation is shown as context only; weak poin...
6,Buyer data,Not used as event-time evidence because availa...
7,Future updates,New cleaner events can be appended to V8-style...


## 13. Executive summary

In [14]:

summary_rows = []

for scenario in SCENARIOS:
    d = final[final["Scenario"].eq(scenario)]

    summary_rows.append({
        "Fee_Increase": scenario,
        "Decision_Rows": len(d),
        "Locations": d["Location"].nunique(),

        "Pct_RPU_Likely_or_Leans_Increase":
            d["RPU_Direction"].isin(
                ["Likely Increase","Leans Increase"]
            ).mean(),

        "Median_RPU_Effect": d["RPU_Median"].median(),
        "Median_RPU_P10": d["RPU_P10"].median(),
        "Median_RPU_P90": d["RPU_P90"].median(),

        "Pct_Low_Conversion_Risk":
            (d["Conversion_Risk"]=="Low").mean(),

        "Pct_Moderate_Conversion_Risk":
            (d["Conversion_Risk"]=="Moderate").mean(),

        "Pct_High_Conversion_Risk":
            (d["Conversion_Risk"]=="High").mean(),

        "Pct_Low_Volume_Risk":
            (d["Volume_Risk"]=="Low").mean(),

        "Pct_Moderate_Volume_Risk":
            (d["Volume_Risk"]=="Moderate").mean(),

        "Pct_High_Volume_Risk":
            (d["Volume_Risk"]=="High").mean(),

        "Pct_Strong_or_Moderate_Evidence":
            d["Evidence_Strength"].isin(
                ["Strong","Moderate"]
            ).mean(),

        "Most_Common_Evidence_Level":
            d["Evidence_Level_Used"].mode().iloc[0]
            if len(d) else np.nan,
    })

executive = pd.DataFrame(summary_rows)

executive.to_csv(
    OUTPUT_DIR / "06_V14_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("="*80)
print("V14 COMPLETE")
print("="*80)
print("\nOutput folder:")
print(OUTPUT_DIR)

print("\nPrimary Pricing file:")
print(OUTPUT_DIR / "04_V14_PRICING_REVIEW_TABLE.csv")

print("\nDetailed audit file:")
print(OUTPUT_DIR / "01_V14_PRICING_DECISION_TABLE.csv")

display(executive)


V14 COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v14_outputs

Primary Pricing file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v14_outputs\04_V14_PRICING_REVIEW_TABLE.csv

Detailed audit file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v14_outputs\01_V14_PRICING_DECISION_TABLE.csv


,Fee_Increase,Decision_Rows,Locations,Pct_RPU_Likely_or_Leans_Increase,Median_RPU_Effect,Median_RPU_P10,Median_RPU_P90,Pct_Low_Conversion_Risk,Pct_Moderate_Conversion_Risk,Pct_High_Conversion_Risk,Pct_Low_Volume_Risk,Pct_Moderate_Volume_Risk,Pct_High_Volume_Risk,Pct_Strong_or_Moderate_Evidence,Most_Common_Evidence_Level
0,10,509,24,0.805501,9.141934,-31.285052,49.543294,0.074656,0.715128,0.210216,0.039293,0.675835,0.284872,0.681729,Price Tier
1,15,509,24,1.000000,11.170682,-42.530453,42.634438,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,1.000000,Portfolio Scenario
2,20,509,24,1.000000,24.466728,-28.308010,102.750672,0.000000,0.962672,0.037328,0.017682,0.943026,0.039293,0.944990,Portfolio Scenario



## V14 outputs to share for review

Please send these five files after the notebook completes:

1. `06_V14_EXECUTIVE_SUMMARY.csv`
2. `02_V14_SCENARIO_SUMMARY.csv`
3. `03_V14_EVIDENCE_HIERARCHY_USAGE.csv`
4. `04_V14_PRICING_REVIEW_TABLE.csv`
5. `01_V14_PRICING_DECISION_TABLE.csv`

### Primary Pricing output
`04_V14_PRICING_REVIEW_TABLE.csv` is intentionally the simpler business-facing table.

### Detailed audit output
`01_V14_PRICING_DECISION_TABLE.csv` retains the underlying empirical downside frequencies, validation context, and hierarchy trail so the result is explainable.
